In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3
!pip -q install speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/render_frontend_conditions.py") if "persian-voice-frontend-listening-v2" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/render_frontend_conditions.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-ezafe-voices-run-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def fetch(plan, minutes, flags=()):
    """scripts/fetch_inputs.py under run_bounded: every request timed out, the stage bounded."""
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text(json.dumps(plan))
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *flags], timeout=minutes * 60, log=LOGS / "fetch.log",
                      monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    return run.ok and report.get("ok") is True, fetched

(REPO / "eval/runs/ezafe-voices-v1/texts.json").parent.mkdir(parents=True, exist_ok=True)
(REPO / "eval/runs/ezafe-voices-v1/texts.json").write_text('{\n "version": 1,\n "purpose": "the ezafe voices check: the ezafe check\'s twelve sentences as the app writes them, read by every form for every voice",\n "from": "eval/runs/ezafe-check-v1/texts.json (14b79becbc2a91c1...), each sentence\'s one text for A1, E1 and E5",\n "texts": [\n  {\n   "key": "e01",\n   "text": "وزارتِ نیرو اعلام کَرد مصرفِ برقِ خانگی در تابستانِ امسال هشت درصد کمتر از پارسال بوده است."\n  },\n  {\n   "key": "e02",\n   "text": "مأمورانِ گمرکِ بندرعباس هفتهٔ گذشته محمولهٔ بزرگی از لوازمِ خانگیِ قاچاق را کشف کردند."\n  },\n  {\n   "key": "e03",\n   "text": "تیمِ ملیِ والیبالِ ایران شبِ گذشته در دیداری نفس\u200cگیر حریفِ ژاپنیِ خود را شکست داد."\n  },\n  {\n   "key": "e04",\n   "text": "هفتهٔ پیش قیمتِ خودروهایِ داخلی در بازارِ تهران دوباره بالا رفت."\n  },\n  {\n   "key": "e05",\n   "text": "سازمانِ هواشناسی برایِ روزهایِ پایانیِ هفته در استان\u200cهایِ شمالی بارانِ شَدید پیش\u200cبینی کرده است."\n  },\n  {\n   "key": "e06",\n   "text": "پزشکانِ بیمارستانِ کودکانِ شیراز دیروز نخستین عملِ پیوندِ قلبِ این استان را با موفقیت انجام دادند."\n  },\n  {\n   "key": "e07",\n   "text": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهایِ بالکن را آب می\u200cدهد."\n  },\n  {\n   "key": "e08",\n   "text": "پسرِ همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد."\n  },\n  {\n   "key": "e09",\n   "text": "بَعد از شام پدربزرگ برایِ همه چای ریخت و از خاطره\u200cهایِ جوانی\u200cاش گفت."\n  },\n  {\n   "key": "e10",\n   "text": "امروز عصر خواهرِ کوچکم نقاشیِ تازه\u200cاش را با ذوق به همه نشان داد."\n  },\n  {\n   "key": "e11",\n   "text": "رانندهٔ اتوبوس وسطِ راه ایستاد و از مسافران خواست کمربندهایشان را ببندند."\n  },\n  {\n   "key": "e12",\n   "text": "دوستم سالِ گذشته مغازهٔ کوچکی در بازار باز کَرد و حالا مشتری\u200cهایِ زیادی دارد."\n  }\n ],\n "sha256_of_texts": "e47a3c3c3771e27b873210290f1cbd26cde70bcfeaf989269fa42301c1caf78c"\n}\n', encoding="utf-8")
(REPO / "scripts/prepare_valid_corpus.py").parent.mkdir(parents=True, exist_ok=True)
(REPO / "scripts/prepare_valid_corpus.py").write_text('#!/usr/bin/env python3\n"""Write Neyshekar\'s validation speakers as a benchmark corpus, for a check that may choose.\n\nThe sealed test sets (``scripts/prepare_test_corpus.py``) are scored once per\nfinal candidate and never used to choose. A check that chooses between exports\non many voices needs speakers of its own. Neyshekar v6\'s validation split has\n26: in no training split of the adaptation (their clips gave only its\nvalidation loss), not on OmniVoice\'s published list of training data, and never\nused to choose anything here.\n\nThe roles are the test sets\' rule, decided on durations alone: the reference is\nthe speaker\'s longest clip of 6.5 to 11.4 s, and the four longest of the rest,\nof at least 3.5 s, are held out. A speaker without them, or with a clip missing\nfrom the parquet or unreadable, is left out and listed; there is no reserve.\n\nOutput as the test sets\': ``<out>/manifest.json`` in the benchmark\'s format\n(``nev01_000.wav`` is the reference, ``_001`` to ``_004`` the held-out clips),\neach clip\'s bytes as the parquet stores them, and ``<out>/roster.json`` with who\nis in, who was left out and why, and every file\'s sha256. Only validation\nrecords are looked up.\n\n    .venv/bin/python scripts/prepare_valid_corpus.py --splits eval/manifests/adaptation_splits_v1.jsonl.gz \\\\\n        --parquet-dir /tmp/neyshekar-validation --out /tmp/valid-ney\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom prepare_test_corpus import ROLE_RULE, load_records, read_audio, roster, whole_reference_roles  # noqa: E402\n\nSOURCE = "neyshekar"\nSPLIT = "valid"\nPREFIX = "nev"\nCORPUS = "neyshekar_v6_validation"\n\n\ndef validation(records: list[dict[str, Any]]) -> tuple[list[str], dict[str, list[dict]]]:\n    clips: dict[str, list[dict]] = defaultdict(list)\n    for record in records:\n        if record["source"] == SOURCE and record["split"] == SPLIT:\n            clips[record["speaker"]].append(record)\n    return sorted(clips), dict(clips)\n\n\ndef write(result: dict[str, Any], audio: dict[str, bytes], out: Path, splits_sha256: str) -> dict[str, Any]:\n    out.mkdir(parents=True, exist_ok=True)\n    entries, files, speakers = [], {}, []\n    for index, row in enumerate(result["chosen"], start=1):\n        label = f"{PREFIX}{index:02d}"\n        reference, held = row["roles"]\n        for position, record in enumerate((reference, *held)):\n            name = f"{label}_{position:03d}.wav"\n            data = audio[record["clip"]]\n            (out / name).write_bytes(data)\n            files[name] = hashlib.sha256(data).hexdigest()\n            entries.append({\n                "speaker": label, "file": name, "role": "reference" if position == 0 else "held_out",\n                "transcript": record["text"].strip(), "duration": record["duration"],\n                "gender": record["gender"], "corpus": CORPUS,\n                "session": record["clip"], "speaker_hash": row["speaker"], "source_ref": record.get("source_ref"),\n            })\n        speakers.append({"label": label, "speaker": row["speaker"], "gender": reference["gender"],\n                         "reference": reference["clip"], "reference_seconds": reference["duration"],\n                         "held_out": [c["clip"] for c in held]})\n    manifest = {\n        "version": 1,\n        "purpose": "Neyshekar\'s validation speakers: a set that may choose, kept apart from the sealed test sets",\n        "speakers": len(speakers),\n        "clips": len(entries),\n        "sources": [CORPUS],\n        "notes": ROLE_RULE,\n        "entries": entries,\n    }\n    (out / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    record = {\n        "source": SOURCE,\n        "split": SPLIT,\n        "splits_sha256": splits_sha256,\n        "role_rule": ROLE_RULE,\n        "speakers": speakers,\n        "left_out": result["left_out"],\n        "files": files,\n        "manifest_sha256": hashlib.sha256((out / "manifest.json").read_bytes()).hexdigest(),\n    }\n    (out / "roster.json").write_text(json.dumps(record, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    return record\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    parser.add_argument("--splits", type=Path, required=True)\n    parser.add_argument("--parquet-dir", type=Path, required=True)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args(argv)\n    records = load_records(args.splits)\n    speakers, clips = validation(records)\n    wanted = {}\n    for rows in clips.values():\n        roles = whole_reference_roles(rows)\n        if roles is not None:\n            wanted.update({r["clip"]: r for r in (roles[0], *roles[1])})\n    audio = read_audio(SOURCE, args.parquet_dir, wanted)\n    result = roster(SOURCE, speakers, {}, clips, audio)\n    record = write(result, audio, args.out, hashlib.sha256(args.splits.read_bytes()).hexdigest())\n    print(f"{len(record[\'speakers\'])} of {len(speakers)} validation speakers written to {args.out}; "\n          f"left out: {record[\'left_out\']}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
NEEDED = ['scripts/benchmark.py', 'scripts/prepare_test_corpus.py', 'scripts/build_adaptation_splits.py', 'scripts/prepare_adaptation_data.py', 'scripts/omnivoice_worker.py', 'eval/runs/ezafe-voices-v1/texts.json', 'scripts/prepare_valid_corpus.py']
missing = [name for name in NEEDED if not (REPO / name).is_file()]
if "--sentences-file" not in (REPO / "scripts/benchmark.py").read_text():
    missing.append("benchmark.py --sentences-file")
REGISTERED = {'eval/runs/ezafe-voices-v1/texts.json': '47f2616062622a4c88eb1ffd93699f8243cc5fa55c58908c805c30313afc0dfe', 'scripts/prepare_valid_corpus.py': 'c177a170370e49a1958f2cb93c7ceb7a20844d7cef2ef5f89d8567b709d733a5'}
for name, digest in REGISTERED.items():
    if (REPO / name).is_file() and hashlib.sha256((REPO / name).read_bytes()).hexdigest() != digest:
        missing.append(f"{name} is not the registered one")
GATES.record("code", not missing, f"missing or changed: {missing}" if missing else
             "the benchmark, the corpus code and the registered texts are present; code from " +
             os.environ["VOICECLONE_CODE_COMMIT"])
if not GATES.skip("download", ("runner", "code")):
    fetched_ok, fetched = fetch({'snapshots': [{'name': 'base', 'repo': 'k2-fsa/OmniVoice', 'revision': 'c5fdb5ccb189668d56333f77ba2629f4cd7535f4', 'local_dir': '/tmp/omnivoice-base', 'sha256': {'audio_tokenizer/model.safetensors': 'fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d'}}, {'name': 'neyshekar validation', 'repo': 'shekar-ai/neyshekar', 'repo_type': 'dataset', 'revision': '30696ada746a66bb3047c984744b5ecfd60fe4e9', 'allow_patterns': ['data/validation-*'], 'local_dir': '/tmp/neyshekar-validation', 'forbid': ['train-*', 'test-*']}]}, 30, ['--no-xet'])
    GATES.record("download", fetched_ok, json.dumps(fetched))
# The texts carry the app's marks; the dataset's tagger must not add its own.
os.environ["VOICECLONE__TEXT__EZAFE__ENABLED"] = "false"
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"

MODELS = {'A1': 'export-1000-float16', 'E1': 'export-1000-float16', 'E5': 'export-5000-float16'}
SESSIONS = {'A1': 'persian-voice-omnivoice-train-a', 'E1': 'persian-voice-omnivoice-ezafe-a', 'E5': 'persian-voice-omnivoice-ezafe-c'}
KNOWN_SHA256 = {'A1': '9c453a74e9eab8324c27e039b9bdc052cd8d6721cecbb20df53c919a0e6ba39a', 'E1': '136aac641d2e04c24c8c081bcefd8ceccf310969b065ef510a5fc2903d720e2b', 'E5': '1ecbaa58e2cf3efb753714818c974f1027b61c1564e54b489fc5f23af3595a4e'}
WEIGHTS = {}
if not GATES.skip("weights", ("download",)):
    problems = []
    for form, name in MODELS.items():
        if name is None:
            WEIGHTS[form] = {"path": "/tmp/omnivoice-base"}
            continue
        hits = sorted(p for p in INPUT.rglob(name) if p.is_dir() and (p / "model.safetensors").is_file() and SESSIONS[form] in p.parts)
        if not hits:
            problems.append(f"{form}: no {name} with a model.safetensors under /kaggle/input")
            continue
        digest = hashlib.sha256()
        with open(hits[0] / "model.safetensors", "rb") as handle:
            for block in iter(lambda: handle.read(1 << 24), b""):
                digest.update(block)
        WEIGHTS[form] = {"path": str(hits[0]), "found": [str(p) for p in hits], "model_sha256": digest.hexdigest()}
        if form in KNOWN_SHA256 and digest.hexdigest() != KNOWN_SHA256[form]:
            problems.append(f"{form}: model.safetensors is not the one recorded before")
    GATES.record("weights", not problems, json.dumps({"weights": WEIGHTS, "problems": problems}))


In [ ]:
CORPUS = Path("/tmp/valid-ney")
EXPECTED_CORPUS = {'manifest_sha256': '47b8b569d8b130fce7e89b3ae230bac003dd25d13ac46aee939922456101c944', 'files_sha256': '9131dc933ea66d4853160d1e7187d9c4f9414227be2040a73b055d50e58ffcc5'}
ROSTER = {}
if not GATES.skip("corpus", ("download",)):
    run = run_bounded([sys.executable, str(REPO / "scripts/prepare_valid_corpus.py"), "--splits", str(SPLITS),
                       "--parquet-dir", "/tmp/neyshekar-validation", "--out", str(CORPUS)],
                      timeout=30 * 60, log=LOGS / "corpus.log", monitor_every=300)
    RUNS["corpus"] = run.record()
    print(run.summary())
    ROSTER = json.loads((CORPUS / "roster.json").read_text()) if (CORPUS / "roster.json").is_file() else {}
    got = {"manifest_sha256": ROSTER.get("manifest_sha256"),
           "files_sha256": hashlib.sha256(json.dumps(ROSTER.get("files", {}), sort_keys=True).encode()).hexdigest()}
    GATES.record("corpus", run.ok and got == EXPECTED_CORPUS and len(ROSTER.get("speakers", [])) == 25,
                 json.dumps({"speakers": len(ROSTER.get("speakers", [])), "left_out": ROSTER.get("left_out"), **got}))


In [ ]:
LABELS = {form: f"ezafe-voices-{form}" for form in MODELS}
SEEDS = [42]
RESULT = {}
for index, form in enumerate(MODELS):
    gate = f"bench-{form}"
    if GATES.skip(gate, ("weights", "corpus")):
        continue
    os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = WEIGHTS[form]["path"]
    run = run_bounded([sys.executable, "scripts/benchmark.py", "--label", LABELS[form], "--engine", "omnivoice",
                       "--corpus-dir", str(CORPUS), "--sentences-file", str(REPO / "eval/runs/ezafe-voices-v1/texts.json"),
                       "--seeds", *map(str, SEEDS), "--device", "cuda", *(["--asr-floor"] if index == 0 else [])],
                      cwd=REPO, timeout=2.5 * 3600, log=LOGS / f"{LABELS[form]}.log", monitor_every=600)
    RUNS[gate] = run.record()
    print(run.summary())
    path = REPO / "eval/results" / f"{LABELS[form]}.json"
    data = json.loads(path.read_text()) if path.is_file() else {}
    summary = data.get("summary", {})
    models = sorted({str((row.get("metadata") or {}).get("model")) for row in data.get("rows", [])})
    RESULT[form] = {**{k: summary.get(k) for k in ("renders", "failures", "held_out_mean", "cer_mean")},
                    "models_in_the_renders": models}
    GATES.record(gate, run.ok and summary.get("failures", 1) == 0
                 and summary.get("renders", 0) == 25 * 12 * len(SEEDS)
                 and models == [WEIGHTS[form]["path"]], json.dumps(RESULT[form], ensure_ascii=False))
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"


In [ ]:
destination = Path("/kaggle/working/ezafe-voices-v1")
(destination / "results").mkdir(parents=True, exist_ok=True)
(destination / "corpus").mkdir(exist_ok=True)
for form, label in LABELS.items():
    if (REPO / "eval/results" / f"{label}.json").is_file():
        shutil.copy2(REPO / "eval/results" / f"{label}.json", destination / "results" / f"{label}.json")
    if (REPO / "outputs/benchmark" / label).is_dir():
        shutil.copytree(REPO / "outputs/benchmark" / label, destination / "renders" / label, dirs_exist_ok=True)
for name in ("manifest.json", "roster.json"):
    if (CORPUS / name).is_file():
        shutil.copy2(CORPUS / name, destination / "corpus" / name)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
(destination / "result.json").write_text(json.dumps({"bench": RESULT, "weights": WEIGHTS}, indent=2,
                                                    ensure_ascii=False))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
print("gates:", json.dumps(GATES.summary(), indent=1))
